# Embeddings & Similarity — minimal demo

A scratch setup for feeling out what the embedding model does and does **not** capture, using the
real client (`clients/openai_client.py`, `settings.openai.EMBEDDING_MODEL`).

Three little experiments:
1. **Genres (fiction & non-fiction)** — do bare labels, then richer *descriptions*, cluster the way
   we'd expect, and do the two groups separate?
2. **Author names** — do names carry usable semantic signal, or is this really a string-match job?
3. **`semantic_input` vs a book document** — embed `"title + author + metadata + description"` and
   score it against a query like `"find books with 400 pages in 1990"`. The point is to *see*
   that numeric/structured constraints (page count, year) don't ride along in the vector — which is
   exactly why `Analyze_Recommend` splits `semantic_input` (theme/tone) from `filters` (metadata).

> Run from `backend/` so `from config import settings` resolves. Needs a valid `OPENAI` key in `config/.env`.

In [10]:
import numpy as np
import pandas as pd

from config import settings
from clients.openai_client import OpenAIClient

llm_client = OpenAIClient(openai_settings=settings.openai)
print("model:", llm_client.embedding_model, "| dims:", llm_client.embedding_dimensions)

model: text-embedding-3-large | dims: 1024


In [11]:
# --- tiny helpers -----------------------------------------------------------
# get_embeddings is async; Jupyter supports top-level await, so `await embed(...)`.

async def embed(texts: list[str]) -> np.ndarray:
    """Embed a list of strings -> (n, dims) float array."""
    vecs = await llm_client.get_embeddings(texts)
    return np.asarray(vecs, dtype=np.float32)


def cosine(a: np.ndarray, b: np.ndarray) -> np.ndarray:
    """Cosine similarity. Same 1 - cosine_distance convention as
    db/stores/utils.build_embedding_search. Returns (len(a), len(b))."""
    a = a / np.linalg.norm(a, axis=-1, keepdims=True)
    b = b / np.linalg.norm(b, axis=-1, keepdims=True)
    return a @ b.T


def sim_frame(labels_a, labels_b, mat) -> pd.DataFrame:
    return pd.DataFrame(mat, index=labels_a, columns=labels_b).round(3)


def rank(query_vec: np.ndarray, doc_vecs: np.ndarray, labels: list[str]) -> pd.DataFrame:
    scores = cosine(query_vec[None, :], doc_vecs)[0]
    return (
        pd.DataFrame({"doc": labels, "similarity": scores.round(3)})
        .sort_values("similarity", ascending=False)
        .reset_index(drop=True)
    )

## 1. Genres — fiction & non-fiction

Two passes. First the bare one-word-ish labels, just to see the raw matrix. Then the richer version:
give each genre a one-line **description + example titles** and embed *that*. Expect the description
embeddings to cluster — the six fiction genres pulling together, the four non-fiction genres pulling
together, and the two groups sitting apart.

In [12]:
# First pass: bare one-word-ish labels.
genres = [
    "fantasy", "science fiction", "mystery", "romance", "historical fiction", "thriller",
    "biography", "true crime", "self-help", "history",
]

g_vecs = await embed(genres)
sim_frame(genres, genres, cosine(g_vecs, g_vecs))

,fantasy,science fiction,mystery,romance,historical fiction,thriller,biography,true crime,self-help,history
fantasy,1.000,0.598,0.478,0.547,0.447,0.444,0.353,0.347,0.287,0.309
science fiction,0.598,1.000,0.420,0.424,0.545,0.451,0.332,0.412,0.266,0.301
mystery,0.478,0.420,1.000,0.468,0.356,0.535,0.351,0.440,0.247,0.332
romance,0.547,0.424,0.468,1.000,0.439,0.443,0.395,0.360,0.269,0.333
historical fiction,0.447,0.545,0.356,0.439,1.000,0.326,0.402,0.403,0.236,0.445
thriller,0.444,0.451,0.535,0.443,0.326,1.000,0.382,0.477,0.224,0.229
biography,0.353,0.332,0.351,0.395,0.402,0.382,1.000,0.374,0.328,0.497
true crime,0.347,0.412,0.440,0.360,0.403,0.477,0.374,1.000,0.247,0.307
self-help,0.287,0.266,0.247,0.269,0.236,0.224,0.328,0.247,1.000,0.198
history,0.309,0.301,0.332,0.333,0.445,0.229,0.497,0.307,0.198,1.000


In [13]:
# Second pass: a one-line description + example titles per genre. Embed the *description*.
GENRES = {
    # --- Fiction ---
    "Fantasy": "Magic and supernatural elements, often set in invented worlds. Examples: The Lord of the Rings, A Game of Thrones.",
    "Science Fiction": "Futuristic technology, space travel, and scientific concepts. Examples: Dune, The Martian.",
    "Mystery": "A protagonist investigates a crime or puzzle. Examples: Murder on the Orient Express, The Hound of the Baskervilles.",
    "Romance": "Centers on a love story with an emotionally satisfying ending. Examples: The Notebook, Pride and Prejudice.",
    "Historical Fiction": "Fictional stories set against real, past historical events. Examples: The Book Thief, All the Light We Cannot See.",
    "Thriller": "Fast-paced suspense, danger, and high-stakes conflict. Examples: Gone Girl, The Da Vinci Code.",
    # --- Non-fiction ---
    "Biography & Memoir": "A written account of a person's life; a memoir focuses on a period, usually self-written. Examples: The Diary of a Young Girl, Educated.",
    "True Crime": "Factual accounts of real crimes, investigations, and trials. Examples: In Cold Blood, Killers of the Flower Moon.",
    "Self-Help & How-To": "Guides to improve one's personal or professional life. Examples: Atomic Habits, The 7 Habits of Highly Effective People.",
    "History": "Factual, scholarly, or narrative explorations of past events. Examples: Sapiens, A People's History of the United States.",
}

KIND = {  # ground-truth grouping, used by the clustering check below
    "Fantasy": "fiction", "Science Fiction": "fiction", "Mystery": "fiction",
    "Romance": "fiction", "Historical Fiction": "fiction", "Thriller": "fiction",
    "Biography & Memoir": "non-fiction", "True Crime": "non-fiction",
    "Self-Help & How-To": "non-fiction", "History": "non-fiction",
}

genre_labels = list(GENRES)
desc_vecs = await embed(list(GENRES.values()))
sim_frame(genre_labels, genre_labels, cosine(desc_vecs, desc_vecs))

,Fantasy,Science Fiction,Mystery,Romance,Historical Fiction,Thriller,Biography & Memoir,True Crime,Self-Help & How-To,History
Fantasy,1.000,0.560,0.394,0.338,0.508,0.451,0.224,0.315,0.249,0.362
Science Fiction,0.560,1.000,0.369,0.288,0.435,0.469,0.203,0.309,0.278,0.417
Mystery,0.394,0.369,1.000,0.367,0.329,0.514,0.222,0.451,0.250,0.298
Romance,0.338,0.288,0.367,1.000,0.397,0.415,0.308,0.246,0.276,0.282
Historical Fiction,0.508,0.435,0.329,0.397,1.000,0.435,0.372,0.469,0.232,0.557
Thriller,0.451,0.469,0.514,0.415,0.435,1.000,0.227,0.429,0.311,0.309
Biography & Memoir,0.224,0.203,0.222,0.308,0.372,0.227,1.000,0.373,0.352,0.419
True Crime,0.315,0.309,0.451,0.246,0.469,0.429,0.373,1.000,0.275,0.469
Self-Help & How-To,0.249,0.278,0.250,0.276,0.232,0.311,0.352,0.275,1.000,0.373
History,0.362,0.417,0.298,0.282,0.557,0.309,0.419,0.469,0.373,1.000


In [14]:
# Does the description embedding actually separate fiction from non-fiction?
# Compare average similarity *within* a group against *across* groups.
S = cosine(desc_vecs, desc_vecs)
kinds = np.array([KIND[g] for g in genre_labels])

def block_mean(mask: np.ndarray) -> float:
    vals = S[np.ix_(mask, mask)]
    off_diag = vals[~np.eye(vals.shape[0], dtype=bool)]  # drop the 1.0 self-similarity
    return float(off_diag.mean())

fic, non = kinds == "fiction", kinds == "non-fiction"
print(f"avg sim within fiction     : {block_mean(fic):.3f}")
print(f"avg sim within non-fiction : {block_mean(non):.3f}")
print(f"avg sim across the two     : {S[np.ix_(fic, non)].mean():.3f}")

avg sim within fiction     : 0.418
avg sim within non-fiction : 0.377
avg sim across the two     : 0.316


In [15]:
# Free-text label -> nearest genre, scored against the description embeddings.
for q in ["spooky", "a rags-to-riches true story", "colonizing another planet", "how to build better habits"]:
    qv = (await embed([q]))[0]
    top = rank(qv, desc_vecs, genre_labels).iloc[0]
    print(f"{q!r:32} -> {top['doc']} ({top['similarity']})")

'spooky'                         -> Thriller (0.296999990940094)
'a rags-to-riches true story'    -> Romance (0.2849999964237213)
'colonizing another planet'      -> Science Fiction (0.3310000002384186)
'how to build better habits'     -> Self-Help & How-To (0.46799999475479126)


## 2. Author names

Names are mostly opaque tokens to the embedder — it has *some* world knowledge, but two authors in
the same genre won't reliably look similar, and a typo won't sit close to the correct spelling.
Takeaway to confirm below: author lookup wants **exact / fuzzy string match** (what `Retrieve_by_Author`
does), not vector similarity.

In [16]:
authors = [
    "Brandon Sanderson",
    "Robert Jordan",       # same genre (epic fantasy) as Sanderson
    "Agatha Christie",
    "Isaac Asimov",
    "Brandon Sandrson",    # typo of the first — will vector-sim catch it?
]

a_vecs = await embed(authors)
sim_frame(authors, authors, cosine(a_vecs, a_vecs))

,Brandon Sanderson,Robert Jordan,Agatha Christie,Isaac Asimov,Brandon Sandrson
Brandon Sanderson,1.000,0.506,0.245,0.367,0.600
Robert Jordan,0.506,1.000,0.242,0.217,0.385
Agatha Christie,0.245,0.242,1.000,0.366,0.088
Isaac Asimov,0.367,0.217,0.366,1.000,0.170
Brandon Sandrson,0.600,0.385,0.088,0.170,1.000


## 3. `semantic_input` vs a book document

Build the kind of text we'd embed per book — `title + author + metadata + description` — for a few
toy books, then score a structured-sounding query against them.

Watch what wins: ranking is driven by the **description prose**, not by the literal `400 pages` /
`1990`. Numbers barely move cosine similarity. That's the demo's whole point — page count and year
are `filters` (SQL `WHERE` / `BooksFilter`), while `semantic_input` should carry only theme/tone.

In [17]:
books = [
    {
        "title": "The Silent Tide",
        "authors": "Maren Holt",
        "categories": "Literary Fiction",
        "published_year": 1990,
        "num_pages": 402,
        "description": "A quiet coastal town confronts a slow ecological collapse over one long summer.",
    },
    {
        "title": "Neon Divide",
        "authors": "Cass Ryel",
        "categories": "Science Fiction",
        "published_year": 2019,
        "num_pages": 390,
        "description": "In a fractured megacity, a courier smuggles memories across a militarized border.",
    },
    {
        "title": "The Long Field",
        "authors": "Ada Perrin",
        "categories": "Historical Fiction",
        "published_year": 1991,
        "num_pages": 405,
        "description": "Three generations of a farming family weather the decade of change after the war.",
    },
    {
        "title": "Paper Moons",
        "authors": "Ilse Von",
        "categories": "Romance",
        "published_year": 2005,
        "num_pages": 210,
        "description": "Two rival illustrators fall for each other while ghost-drawing the same comic strip.",
    },
]


def book_document(b: dict) -> str:
    """The per-book text we embed. Mirrors the idea of
    db/ingestion/embeddings._get_embedding_text but with metadata folded in."""
    return (
        f"{b['title']}\n"
        f"by {b['authors']}\n"
        f"{b['categories']}, {b['published_year']}, {b['num_pages']} pages\n\n"
        f"{b['description']}"
    )


docs = [book_document(b) for b in books]
print(docs[0])
doc_vecs = await embed(docs)

The Silent Tide
by Maren Holt
Literary Fiction, 1990, 402 pages

A quiet coastal town confronts a slow ecological collapse over one long summer.


In [18]:
doc_vecs

array([[-0.02587891, -0.04373169, -0.02539062, ..., -0.04818726,
         0.02046204, -0.03497314],
       [-0.00025415, -0.04931641, -0.03344727, ...,  0.00288963,
         0.01148224, -0.01442719],
       [-0.00296974, -0.03765869, -0.02658081, ..., -0.06707764,
         0.08087158,  0.00590134],
       [ 0.00373459, -0.03692627, -0.03050232, ..., -0.01966858,
         0.03787231, -0.03619385]], shape=(4, 1024), dtype=float32)

In [19]:
# A structured-sounding query. The 402-page 1990 book (The Silent Tide) is the
# "correct" metadata answer — see whether similarity actually surfaces it.
semantic_input = "find books with 400 pages in 1990"
q_vec = (await embed([semantic_input]))[0]
rank(q_vec, doc_vecs, [b["title"] for b in books])

,doc,similarity
0,The Silent Tide,0.393
1,The Long Field,0.385
2,Paper Moons,0.349
3,Neon Divide,0.317


In [20]:
# Contrast: a genuinely thematic query is what embeddings are good at.
for q in ["find books with 400 pages in 1990", "a dystopian city and surveillance", "a tender love story"]:
    qv = (await embed([q]))[0]
    top = rank(qv, doc_vecs, [b["title"] for b in books]).iloc[0]
    print(f"{q!r:52} -> {top['doc']!r} ({top['similarity']})")

'find books with 400 pages in 1990'                  -> 'The Silent Tide' (0.3930000066757202)
'a dystopian city and surveillance'                  -> 'Neon Divide' (0.4390000104904175)
'a tender love story'                                -> 'Paper Moons' (0.4189999997615814)


### Takeaway

- **Genres** embed sensibly — bare labels already cluster, and richer descriptions separate fiction
  from non-fiction (compare the within- vs across-group averages). Vector similarity is a fine way to
  snap a fuzzy phrase (`"spooky"`, `"how to build better habits"`) onto a known genre.
- **Author names** don't — treat author matching as string/fuzzy lookup, not similarity.
- **Numeric constraints** (`400 pages`, `1990`) don't survive into the embedding: the thematic query
  routes correctly while the metadata query doesn't. So keep `semantic_input` = theme/tone only, and
  push page/year/rating into `filters` — the same split the catalog draws between `Analyze_Recommend`
  and `Filter_Retrieval`.

Next steps when we go past minimal: pull real rows from `books` (with stored `embedding`) and score
against them via `build_embedding_search` instead of these toy docs.